In [ ]:
import os as _o; _o.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import signal, time as _t
_T0 = _t.time()
def _cap(*a): raise TimeoutError("25 min cap hit after %.0fs" % (_t.time() - _T0))
signal.signal(signal.SIGALRM, _cap); signal.alarm(25 * 60)
import subprocess, os, sys, json, time, glob, argparse, base64, io
def sh(c, check=True): print(c, flush=True); return subprocess.run(c, shell=True, check=check).returncode
sh("pip -q install 'transformers==4.57.1' diffusers==0.32.2 peft==0.14.0 gguf>=0.10 sentencepiece protobuf mmengine gdown einops")
PL = "/kaggle/working/CLD"
if not os.path.exists(PL): sh(f"git clone --depth 1 https://github.com/monkek123King/CLD {PL}")
sys.path.insert(0, PL)
_f = f"{PL}/models/multiLayer_adapter.py"   # diffusers 0.32 no longer re-exports BaseOutput from models.controlnet
_t = open(_f).read().replace("from diffusers.models.controlnet import BaseOutput, zero_module", "from diffusers.utils import BaseOutput; from diffusers.models.controlnets.controlnet import zero_module")
open(_f, "w").write(_t)
import torch, torch.nn as nn, numpy as np
from PIL import Image
from huggingface_hub import hf_hub_download, snapshot_download
from safetensors.torch import load_file
from diffusers import FluxTransformer2DModel, GGUFQuantizationConfig, FlowMatchEulerDiscreteScheduler
import diffusers; print("diffusers", diffusers.__version__, flush=True)

# ---- weights ----
gguf = hf_hub_download("city96/FLUX.1-dev-gguf", "flux1-dev-Q4_K_S.gguf")
BASE = snapshot_download("ostris/Flex.1-alpha", allow_patterns=["model_index.json","text_encoder/*","text_encoder_2/*","tokenizer/*","tokenizer_2/*","vae/*","scheduler/*"])
CLD = snapshot_download("thuteam/CLD")
print("CLD files:", sorted(os.path.relpath(f, CLD) for f in glob.glob(f"{CLD}/**/*", recursive=True) if os.path.isfile(f)), flush=True)
ADP = snapshot_download("alimama-creative/FLUX.1-dev-Controlnet-Inpainting-Alpha")
pre_fuse = f"{CLD}/pre_trained_LoRA/pytorch_lora_weights.safetensors"
art_lora = f"{CLD}/prism_ft_LoRA/pytorch_lora_weights.safetensors"
cld_lora = f"{CLD}/decouple_LoRA/transformer/pytorch_lora_weights.safetensors"
adp_lora = f"{CLD}/decouple_LoRA/adapter/pytorch_lora_weights.safetensors"
art_pe = f"{CLD}/decouple_LoRA/layer_pe.pth"
def load_any(p):
    return load_file(p) if p.endswith(".safetensors") else torch.load(p, map_location="cpu")

CFG = "/kaggle/working/flux_dev_cfg"; os.makedirs(f"{CFG}/transformer", exist_ok=True)
json.dump({"_class_name":"FluxTransformer2DModel","attention_head_dim":128,"axes_dims_rope":[16,56,56],"guidance_embeds":True,
           "in_channels":64,"joint_attention_dim":4096,"num_attention_heads":24,"num_layers":19,"num_single_layers":38,
           "patch_size":1,"pooled_projection_dim":768}, open(f"{CFG}/transformer/config.json","w"))
dev_sched = FlowMatchEulerDiscreteScheduler(num_train_timesteps=1000, shift=3.0, use_dynamic_shifting=True, base_shift=0.5,
                                            max_shift=1.15, base_image_seq_len=256, max_image_seq_len=4096)
D0, D1, DT = torch.device("cuda:0"), torch.device("cuda:1"), torch.float16
from models.mmdit import CustomFluxTransformer2DModel
from models.pipeline import CustomFluxPipelineCfgLayer
from models.multiLayer_adapter import MultiLayerAdapter
from models.transp_vae import AutoencoderKLTransformerTraining as CustomVAE

# ---- transformer: GGUF re-classed, layer_pe, three unmerged LoRAs, fp16_mixed ----
t = FluxTransformer2DModel.from_single_file(gguf, config=CFG, subfolder="transformer",
        quantization_config=GGUFQuantizationConfig(compute_dtype=DT), torch_dtype=DT)
t.__class__ = CustomFluxTransformer2DModel
t.register_to_config(max_layer_num=52); t.max_layer_num = 52
pe = load_any(art_pe); pe = pe.get("layer_pe", next(iter(pe.values()))) if isinstance(pe, dict) else pe
t.layer_pe = nn.Parameter(pe.to(torch.float32).view(1, 52, 1, 1, -1), requires_grad=False)
_raw = load_any(art_pe)
print("layer_pe file:", art_pe, "type", type(_raw).__name__, "keys", list(_raw.keys())[:5] if isinstance(_raw, dict) else None,
      "shape", tuple(pe.shape), "norm", round(float(pe.float().norm()), 3), "frame-to-frame diff", round(float((pe.float().view(52, -1)[1] - pe.float().view(52, -1)[0]).abs().mean()), 5), flush=True)
exec(compile('"""Run the Qwen-Image(-Layered) transformer on GPUs without bf16 (Kaggle T4, P100, RTX 20xx) without NaNs.\n\nQwen-Image is trained for bf16; its hidden values exceed the fp16 limit (65504), so a plain fp16 run gives all-NaN\nlatents (ComfyUI also lists Qwen-Image as bf16/fp32 only). This keeps the fast fp16 tensor-core matmuls but holds\neverything else in fp32:\n\n  * every Linear: input scaled down to absmax 32, fp16 matmul, result scaled back in fp32\n    (falls back to a fp32 matmul for that call if the fp16 result is still non-finite)\n  * attention: q/k in fp16 (RMS-normalised, small), v scaled the same way (attention is linear in v)\n  * residual stream, norms, gates, embeddings: fp32\n  * the model\'s output is returned in fp16, so the diffusers pipeline is unchanged\n\nMeasured on Kaggle 2x T4 (Q4_K_S GGUF, 640 px, 4 layers): no NaN, 42 s/step, peak 8.8 / 8.4 GB.\n\n    from layerproto.fp16_mixed import apply_fp16_mixed\n    apply_fp16_mixed(transformer)        # once, after loading; idempotent\n"""\nimport types\n\nimport torch\nimport torch.nn.functional as F\n\nLINEAR_LIMIT = 32.0  # absmax the fp16 matmul input is scaled to (1024 was still too high: NaNs)\nATTN_LIMIT = 32.0\n\n\ndef _dequant(w):\n    if type(w).__name__ == "GGUFParameter":\n        from diffusers.quantizers.gguf.utils import dequantize_gguf_tensor\n        return dequantize_gguf_tensor(w)\n    return w\n\n\ndef _mixed_linear(self, x):\n    x = x.float()\n    s = (x.abs().amax() / LINEAR_LIMIT).clamp_(min=1.0)\n    y = self._fp16_orig_forward((x / s).to(torch.float16)).float()   # = W x/s + b\n    b = getattr(self, "bias", None)\n    if not torch.isfinite(y).all():                                  # rare: this call in fp32\n        y = F.linear(x, _dequant(self.weight).float(), None if b is None else b.float())\n    elif b is None:\n        y = y * s\n    else:\n        b = b.float()\n        y = (y - b) * s + b\n    loras = getattr(self, "_fp16_lora", None)\n    if loras:                                                        # low-rank deltas (stacked LoRAs), same input scaling\n        xs = (x / s).to(torch.float16)\n        for i, (A, B, k) in enumerate(loras):\n            if A.device != x.device:                                 # module was moved (e.g. split over GPUs)\n                A, B = A.to(x.device), B.to(x.device)\n                loras[i] = (A, B, k)\n            y = y + F.linear(F.linear(xs, A), B).float() * (s * k)\n    return y\n\n\ndef _lora_pairs(sd):\n    """{module name: (A [r, in], B [out, r], alpha or None)} for common LoRA key layouts (PEFT / kohya / diffusers)."""\n    out = {}\n    for key in sd:\n        for a_suf, b_suf in ((".lora_A.weight", ".lora_B.weight"), (".lora_down.weight", ".lora_up.weight"),\n                             (".lora_A.default.weight", ".lora_B.default.weight")):\n            if key.endswith(a_suf):\n                base = key[: -len(a_suf)]\n                bkey = base + b_suf\n                if bkey in sd:\n                    alpha = sd.get(base + ".alpha")\n                    name = base\n                    for pre in ("base_model.model.", "diffusion_model.", "transformer.", "model."):\n                        if name.startswith(pre):\n                            name = name[len(pre):]\n                    out[name] = (sd[key], sd[bkey], None if alpha is None else float(alpha))\n    return out\n\n\ndef add_lora(model, path_or_state_dict, scale: float = 1.0):\n    """Attach a LoRA to an fp16_mixed-patched model (works with GGUF-quantised linears, nothing is merged).\n    Returns (matched modules, LoRA modules not found)."""\n    sd = path_or_state_dict\n    if isinstance(sd, str):\n        from safetensors.torch import load_file\n        sd = load_file(sd)\n    pairs = _lora_pairs(sd)\n    mods = dict(model.named_modules())\n    hit, miss = 0, []\n    for name, (A, B, alpha) in pairs.items():\n        m = mods.get(name)\n        if m is None or not hasattr(m, "_fp16_orig_forward"):\n            miss.append(name)\n            continue\n        dev = next((p.device for p in m.parameters()), torch.device("cpu"))\n        r = A.shape[0]\n        k = scale * ((alpha / r) if alpha else 1.0)\n        # append, never replace: ART+ stacks three LoRAs on the same modules (v1-v8 silently kept only the last one)\n        if not hasattr(m, "_fp16_lora"):\n            m._fp16_lora = []\n        m._fp16_lora.append((A.to(dev, torch.float16), B.to(dev, torch.float16), k))\n        hit += 1\n    return hit, miss\n\n\ndef remove_lora(model):\n    for _, m in model.named_modules():\n        if hasattr(m, "_fp16_lora"):\n            del m._fp16_lora\n\n\ndef _patch_attention():\n    # Qwen-Image attention only; other models (FLUX ports with older bundled diffusers) just get the Linear patch\n    try:\n        import diffusers.models.transformers.transformer_qwenimage as tq\n    except ImportError:\n        return\n    if getattr(tq, "_fp16_orig_attn", None) is not None:\n        return\n    orig = tq.dispatch_attention_fn\n\n    def attn_fp16(q, k, v, attn_mask=None, **kw):\n        s = (v.float().abs().amax() / ATTN_LIMIT).clamp_(min=1.0)\n        if attn_mask is not None and attn_mask.is_floating_point():\n            attn_mask = attn_mask.to(torch.float16)\n        o = orig(q.to(torch.float16), k.to(torch.float16), (v.float() / s).to(torch.float16), attn_mask=attn_mask, **kw)\n        return o.float() * s\n\n    tq._fp16_orig_attn = orig\n    tq.dispatch_attention_fn = attn_fp16\n\n\ndef apply_fp16_mixed(model):\n    """Patch a loaded QwenImageTransformer2DModel in place. Returns (linears patched, params moved to fp32)."""\n    _patch_attention()\n    n_lin = n_par = 0\n    for _, m in model.named_modules():\n        if "Linear" in type(m).__name__:\n            if not hasattr(m, "_fp16_orig_forward"):\n                m._fp16_orig_forward = m.forward\n            m.forward = types.MethodType(_mixed_linear, m)\n            n_lin += 1\n        else:\n            for _, p in m.named_parameters(recurse=False):\n                if p.dtype == torch.float16:\n                    p.data = p.data.float()\n                    n_par += 1\n    if not hasattr(model, "_fp16_orig_model_forward"):\n        model._fp16_orig_model_forward = model.forward\n\n        def fwd(*a, **k):\n            o = model._fp16_orig_model_forward(*a, **k)\n            if isinstance(o, tuple):\n                if isinstance(o[0], (list, tuple)):          # ControlNet-style adapters return (list of block samples, ...)\n                    return tuple([x.to(torch.float16) for x in e] if isinstance(e, (list, tuple)) else e for e in o)\n                return (o[0].to(torch.float16),) + tuple(o[1:])\n            if hasattr(o, "sample"):\n                o.sample = o.sample.to(torch.float16)\n            return o\n\n        model.forward = fwd\n    return n_lin, n_par\n', "fp16_mixed", "exec"))
apply_fp16_mixed(t)
for name, path in (("pre_trained", pre_fuse), ("prism_ft", art_lora), ("decouple", cld_lora)):
    hit, miss = add_lora(t, path, 1.0); print(f"LoRA {name}: matched {hit}, missing {len(miss)}", miss[:3], flush=True)
t.to(D0).eval()
adapter = MultiLayerAdapter.from_pretrained(ADP, torch_dtype=DT)
apply_fp16_mixed(adapter)
hit, miss = add_lora(adapter, adp_lora, 1.0); print("LoRA adapter: matched", hit, "missing", len(miss), miss[:3], flush=True)
adapter.set_layerPE(t.layer_pe, t.max_layer_num)
adapter.to(D0).eval()

pipe = CustomFluxPipelineCfgLayer.from_pretrained(BASE, transformer=t, scheduler=dev_sched, torch_dtype=DT)
pipe.set_multiLayerAdapter(adapter); pipe.vae.to(D0, torch.float32); pipe.text_encoder.to(D1); pipe.text_encoder_2.to(D1)
CustomFluxPipelineCfgLayer._execution_device = property(lambda self: D0)
# PrismLayers was written for diffusers 0.31 (FLUX vae_scale_factor 16); 0.32+ uses 8 -> latents for 2x the size, boxes in one corner
pipe.vae_scale_factor = 16; print('vae_scale_factor ->', pipe.vae_scale_factor, flush=True)
_orig_enc = pipe.encode_prompt
def _enc(*a, device=None, **kw):
    out = _orig_enc(*a, device=D1, **kw)
    return tuple(o.to(D0) if torch.is_tensor(o) else o for o in out)
pipe.encode_prompt = _enc

vae_args = argparse.Namespace(max_layers=48, decoder_arch="vit", pos_embedding="rope", layer_embedding="rope", single_layer_decoder=None)
transp_vae = CustomVAE(vae_args)
_tv = glob.glob(f"{CLD}/trans_vae/*.pt")[0]
print("transparent decoder file:", _tv, flush=True)
tv = load_file(_tv) if _tv.endswith(".safetensors") else torch.load(_tv, map_location="cpu", weights_only=False)
print("transp vae:", transp_vae.load_state_dict(tv.get("model", tv) if isinstance(tv, dict) and "model" in tv else tv, strict=False), flush=True)
transp_vae.to(D0, torch.float32).eval()
# VAEs run in fp32 but the pipeline hands them fp16 tensors -> cast tensor inputs (RevealLayer needed the same fix)
def _to32(m, args, kwargs):
    c = lambda x: x.float() if torch.is_tensor(x) and x.is_floating_point() else x
    return tuple(c(a) for a in args), {k: c(v) for k, v in kwargs.items()}
for _v in (transp_vae, pipe.vae):
    _v.register_forward_pre_hook(_to32, with_kwargs=True)
    for _name in ("encode", "decode"):
        if hasattr(_v, _name):
            _f = getattr(_v, _name)
            setattr(_v, _name, (lambda f: lambda *a, **k: f(*[x.float() if torch.is_tensor(x) and x.is_floating_point() else x for x in a], **k))(_f))
# run 4 OOM'd in the fp32 VAE decode with the transformer still on GPU 0: offload it before decoding, slice/tile the VAE
pipe.vae.enable_slicing(); pipe.vae.enable_tiling()
for _v in (transp_vae, pipe.vae):
    _dec = _v.decode
    def _offload_then(*a, _d=_dec, **k):
        if next(t.parameters()).device.type == "cuda":
            t.to("cpu"); adapter.to("cpu"); torch.cuda.empty_cache()
        return _d(*a, **k)
    _v.decode = _offload_then
print("loaded", round(time.time() - _T0), "s; GiB", [round(torch.cuda.memory_allocated(i)/2**30, 1) for i in range(2)], flush=True)

def boxes16(bs):
    return [(((2*x0)//16)*16, ((2*y0)//16)*16, min(1024, (((2*x1)//16)+1)*16), min(1024, (((2*y1)//16)+1)*16)) for x0, y0, x1, y1 in bs]
LAYOUTS = [{"whole_image": {"caption": "Storybook illustration of a thatched stone cottage with a large leafy tree on the right, a stone gate arch on the left, small village houses behind, a cobblestone path leading into the foreground and a red fox standing on the path.", "height": 1024, "width": 1024}, "img": "...", "layout": [[0, 0, 512, 512], [0, 0, 512, 512], [0, 195, 115, 280], [0, 330, 300, 512], [95, 100, 475, 330], [0, 255, 512, 330], [62, 245, 150, 322], [180, 0, 512, 352], [205, 325, 375, 452]]}]
os.makedirs("/kaggle/working/cld_out", exist_ok=True)
for k, inp in enumerate(LAYOUTS):
    for steps in (2,):
        try:
            t.to(D0); adapter.to(D0); torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats(0); t0 = time.time()
            with torch.inference_mode():
                x_hat, image, _lat = pipe(prompt=inp["whole_image"]["caption"], adapter_image=Image.open(io.BytesIO(base64.b64decode(inp["img"]))).convert("RGB"), adapter_conditioning_scale=0.9, validation_box=boxes16(inp["layout"]),
                                       generator=torch.Generator(D0).manual_seed(42), height=1024, width=1024,
                                       num_inference_steps=steps, guidance_scale=4.0, num_layers=len(inp["layout"]), sdxl_vae=transp_vae)
            _l = _lat.float().reshape(_lat.shape[0] * _lat.shape[1] if _lat.dim() == 5 else _lat.shape[0], -1) if torch.is_tensor(_lat) else None
            if _l is not None and _l.shape[0] > 1:
                print("final latents per frame: frame0-vs-frame1 diff", round(float((_l[0] - _l[1]).abs().mean()), 4), "frame std", [round(float(x), 3) for x in _l.std(dim=1)[:6]], flush=True)
            x_hat = ((x_hat + 1) / 2).squeeze(0).permute(1, 0, 2, 3).float()
            d = f"/kaggle/working/cld_out/case{k}_{steps}st"; os.makedirs(d, exist_ok=True)
            image[0].save(f"{d}/whole.png"); image[1].save(f"{d}/background.png")
            merged = image[1].convert("RGBA")
            for i in range(2, x_hat.shape[0]):
                lay = Image.fromarray((x_hat[i].permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8), "RGBA")
                lay.save(f"{d}/layer_{i-2}_rgba.png"); merged = Image.alpha_composite(merged, lay)
            merged.convert("RGB").save(f"{d}/merged.png")
            print(f"case{k} {steps} steps: {time.time()-t0:.0f}s, peak GiB {torch.cuda.max_memory_allocated(0)/2**30:.1f}", flush=True)
        except BaseException as e:
            import traceback; traceback.print_exc(); print("FAILED", k, steps, flush=True)
            if isinstance(e, TimeoutError): raise
print("total", round(time.time() - _T0), "s", flush=True)